# Week 3 - AVM-Safe Data Preprocessing

This notebook prepares California single-family residential sales for the AVM baseline model. It uses chronological train/validation/test splits and excludes known leakage fields.

In [1]:
import json
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Load and combine all monthly CRMLS files

In [2]:
data_dir = Path(r"C:\\Users\\Richa\\OneDrive\\文档\\idx exchange")
output_dir = data_dir / "week3_outputs"
output_dir.mkdir(exist_ok=True)

raw_files = sorted(data_dir.glob("CRMLSSold*.csv"))

if len(raw_files) < 3:
    raise ValueError("At least three complete monthly files are required.")

frames = []

for file_path in raw_files:
    monthly_df = pd.read_csv(file_path, low_memory=False)
    source_month = pd.Period(file_path.stem.replace("CRMLSSold", ""), freq="M")

    monthly_df["SourceFile"] = file_path.name
    monthly_df["SourceMonth"] = str(source_month)
    frames.append(monthly_df)

df = pd.concat(frames, ignore_index=True)

print(f"Loaded {len(raw_files)} monthly files")
print(f"Raw shape: {df.shape}")
print("Months:", sorted(df["SourceMonth"].unique()))

Loaded 8 monthly files
Raw shape: (166725, 80)
Months: ['2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05']


## Data quality log

In [3]:
quality_log = []

def log_step(step_name, before_rows, after_rows):
    quality_log.append({
        "step": step_name,
        "rows_before": before_rows,
        "rows_after": after_rows,
        "rows_removed": before_rows - after_rows,
        "pct_removed_of_input": round(
            ((before_rows - after_rows) / before_rows) * 100, 3
        ) if before_rows else 0
    })

## Filter residential single-family properties

In [4]:
before = len(df)

df = df[
    (df["PropertyType"] == "Residential")
    & (df["PropertySubType"] == "SingleFamilyResidence")
].copy()

log_step("Keep Residential / SingleFamilyResidence only", before, len(df))
print("Filtered shape:", df.shape)

Filtered shape: (83495, 80)


## Remove invalid and duplicate records

In [5]:
# ClosePrice must be present and positive.
before = len(df)
df = df[df["ClosePrice"].notna() & (df["ClosePrice"] > 0)].copy()
log_step("Remove missing or non-positive ClosePrice", before, len(df))

# A zero or negative living area is not physically meaningful.
before = len(df)
df = df[df["LivingArea"].isna() | (df["LivingArea"] > 0)].copy()
log_step("Remove zero or negative LivingArea", before, len(df))

# Remove identical duplicate rows and duplicate Trestle record keys.
before = len(df)
df = df.drop_duplicates().copy()
log_step("Remove exact duplicate rows", before, len(df))

before = len(df)
df = df.drop_duplicates(subset=["ListingKey"], keep="last").copy()
log_step("Remove duplicate ListingKey records", before, len(df))

# A close date cannot be earlier than the listing-contract date.
df["CloseDate_parsed"] = pd.to_datetime(df["CloseDate"], errors="coerce")
df["ListingContractDate_parsed"] = pd.to_datetime(
    df["ListingContractDate"], errors="coerce"
)

invalid_date_order = (
    df["CloseDate_parsed"].notna()
    & df["ListingContractDate_parsed"].notna()
    & (df["CloseDate_parsed"] < df["ListingContractDate_parsed"])
)

before = len(df)
df = df[~invalid_date_order].copy()
log_step("Remove CloseDate earlier than ListingContractDate", before, len(df))

print("Shape after data-quality checks:", df.shape)

Shape after data-quality checks: (83399, 82)


## Create chronological train, validation, and test splits

The latest complete month is reserved as the final test set. The second-most-recent month is used for validation.

In [6]:
months = sorted(df["SourceMonth"].unique())

test_month = months[-1]
validation_month = months[-2]
train_months = months[:-2]

train_df = df[df["SourceMonth"].isin(train_months)].copy()
validation_df = df[df["SourceMonth"] == validation_month].copy()
test_df = df[df["SourceMonth"] == test_month].copy()

print("Train months:", train_months)
print("Validation month:", validation_month)
print("Test month:", test_month)
print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

Train months: ['2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03']
Validation month: 2026-04
Test month: 2026-05
Train shape: (59360, 82)
Validation shape: (12019, 82)
Test shape: (12020, 82)


## Remove ClosePrice outliers using train-only thresholds

In [7]:
lower_price_cutoff = train_df["ClosePrice"].quantile(0.005)
upper_price_cutoff = train_df["ClosePrice"].quantile(0.995)

print(f"Lower cutoff: ${lower_price_cutoff:,.0f}")
print(f"Upper cutoff: ${upper_price_cutoff:,.0f}")

def apply_price_thresholds(split_df):
    return split_df[
        split_df["ClosePrice"].between(
            lower_price_cutoff,
            upper_price_cutoff,
            inclusive="both"
        )
    ].copy()

before = len(train_df)
train_df = apply_price_thresholds(train_df)
log_step("Apply train-derived price thresholds to train", before, len(train_df))

before = len(validation_df)
validation_df = apply_price_thresholds(validation_df)
log_step("Apply frozen train price thresholds to validation", before, len(validation_df))

before = len(test_df)
test_df = apply_price_thresholds(test_df)
log_step("Apply frozen train price thresholds to test", before, len(test_df))

Lower cutoff: $180,000
Upper cutoff: $9,105,125


## Select AVM-safe features

ListPrice, OriginalListPrice, DaysOnMarket, identifiers, agent information, and post-sale fields are excluded because they are not valid inputs for an off-market AVM.

In [8]:
target_col = "ClosePrice"

numeric_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "GarageSpaces",
    "Latitude",
    "Longitude",
]

categorical_features = [
    "City",
    "CountyOrParish",
    "PostalCode",
    "StateOrProvince",
    "FireplaceYN",
    "PoolPrivateYN",
    "ViewYN",
    "AttachedGarageYN",
    "NewConstructionYN",
]

excluded_columns = [
    "ListPrice", "OriginalListPrice", "DaysOnMarket",
    "CloseDate", "CloseDate_parsed", "ListingContractDate",
    "ListingContractDate_parsed", "PurchaseContractDate",
    "ContractStatusChangeDate", "MlsStatus",
    "ListingKey", "ListingKeyNumeric", "ListingId",
    "SourceFile", "SourceMonth",
    "ListAgentEmail", "ListAgentFirstName", "ListAgentLastName",
    "ListAgentFullName", "ListAgentAOR", "BuyerAgentAOR",
    "BuyerAgentMlsId", "BuyerAgentFirstName", "BuyerAgentLastName",
    "BuyerOfficeName", "BuyerOfficeAOR", "ListOfficeName",
    "CoListOfficeName",
]

required_features = numeric_features + categorical_features
missing_features = [c for c in required_features if c not in train_df.columns]

if missing_features:
    raise KeyError(f"Expected feature columns not found: {missing_features}")

for split_df in [train_df, validation_df, test_df]:
    split_df["PostalCode"] = split_df["PostalCode"].astype("string")

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'YearBuilt', 'GarageSpaces', 'Latitude', 'Longitude']
Categorical features: ['City', 'CountyOrParish', 'PostalCode', 'StateOrProvince', 'FireplaceYN', 'PoolPrivateYN', 'ViewYN', 'AttachedGarageYN', 'NewConstructionYN']


## Save clean split datasets and preprocessing configuration

In [9]:
keep_columns = numeric_features + categorical_features + [target_col]

train_clean = train_df[keep_columns].copy()
validation_clean = validation_df[keep_columns].copy()
test_clean = test_df[keep_columns].copy()

train_clean.to_csv(output_dir / "train_clean.csv", index=False)
validation_clean.to_csv(output_dir / "validation_clean.csv", index=False)
test_clean.to_csv(output_dir / "test_clean.csv", index=False)

pd.DataFrame(quality_log).to_csv(
    output_dir / "data_quality_log.csv",
    index=False
)

feature_config = {
    "target": target_col,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "excluded_columns": excluded_columns,
    "train_months": train_months,
    "validation_month": validation_month,
    "test_month": test_month,
    "train_price_lower_cutoff": float(lower_price_cutoff),
    "train_price_upper_cutoff": float(upper_price_cutoff),
}

with open(output_dir / "feature_config.json", "w", encoding="utf-8") as file:
    json.dump(feature_config, file, indent=4)

print("Saved outputs to:", output_dir)
print("Train shape:", train_clean.shape)
print("Validation shape:", validation_clean.shape)
print("Test shape:", test_clean.shape)

pd.DataFrame(quality_log)

Saved outputs to: C:\Users\Richa\OneDrive\文档\idx exchange\week3_outputs
Train shape: (58770, 18)
Validation shape: (11907, 18)
Test shape: (11919, 18)


,step,rows_before,rows_after,rows_removed,pct_removed_of_input
0,Keep Residential / SingleFamilyResidence only,166725,83495,83230,49.921
1,Remove missing or non-positive ClosePrice,83495,83495,0,0.000
2,Remove zero or negative LivingArea,83495,83470,25,0.030
3,Remove exact duplicate rows,83470,83468,2,0.002
4,Remove duplicate ListingKey records,83468,83410,58,0.069
5,Remove CloseDate earlier than ListingContractDate,83410,83399,11,0.013
6,Apply train-derived price thresholds to train,59360,58770,590,0.994
7,Apply frozen train price thresholds to validation,12019,11907,112,0.932
8,Apply frozen train price thresholds to test,12020,11919,101,0.840


## Define the Week 4 leakage-safe preprocessing pipeline

The pipeline is defined here but fitted in Week 4 only on `X_train`, together with the Linear Regression model.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

preprocessor

## Preprocessing summary

- Combined all available CRMLS monthly files.
- Kept residential single-family properties only.
- Removed invalid targets, invalid living areas, duplicates, and invalid date order.
- Used chronological train, validation, and test months.
- Learned ClosePrice outlier thresholds from training data only.
- Excluded listing-price, market-process, identifier, and post-sale leakage fields.
- Saved clean split datasets and an auditable quality log.
- Defined a train-only preprocessing pipeline for Week 4.